In [3]:

import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import pandas as pd
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import re
import string
from rouge_score import rouge_scorer
from rouge_score import scoring
import nltk

import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

[nltk_data] Downloading package punkt to /home/dataconv/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
tokenizer = AutoTokenizer.from_pretrained("McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp")
tokenizer.pad_token = tokenizer.eos_token
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
    quantization_config=quantization_config,
    device_map="auto"
)
model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading adapter weights from McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp led to unexpected keys not found in the model:  ['layers.0.mlp.down_proj.lora_A.default.weight', 'layers.0.mlp.down_proj.lora_B.default.weight', 'layers.0.mlp.gate_proj.lora_A.default.weight', 'layers.0.mlp.gate_proj.lora_B.default.weight', 'layers.0.mlp.up_proj.lora_A.default.weight', 'layers.0.mlp.up_proj.lora_B.default.weight', 'layers.0.self_attn.k_proj.lora_A.default.weight', 'layers.0.self_attn.k_proj.lora_B.default.weight', 'layers.0.self_attn.o_proj.lora_A.default.weight', 'layers.0.self_attn.o_proj.lora_B.default.weight', 'layers.0.self_attn.q_proj.lora_A.default.weight', 'layers.0.self_attn.q_proj.lora_B.default.weight', 'layers.0.self_attn.v_proj.lora_A.default.weight', 'layers.0.self_attn.v_proj.lora_B.default.weight', 'layers.1.mlp.down_proj.lora_A.default.weight', 'layers.1.mlp.down_proj.lora_B.default.weight', 'layers.1.mlp.gate_proj.lora_A.default.weight', 'layers.1.mlp.gate_proj.lora_B.defaul

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
  

In [5]:
# read test data 
df = pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/qa_test.csv')
df.head()

,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,5cf39741-d5bd-4226-a94d-ff0c13ca5eaa,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,41196666-5ad7-4651-98aa-9fa9ddb4aad5,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,6b6a74ae-4b78-4a17-a338-5418ac1408cb,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,e94de938-6049-4e4f-9ac1-b6f98884e235,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,2943f760-b273-4602-bd66-6e5c73ae0edf,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [29]:
question = df.loc[0, 'question']
long_answers = eval(df.loc[0, 'long_answers'])
prompt_template = 'Please summarize the two answers to the question "{question}" into one answer.\n\nAnswer 1:\n{long_answers[0]}\n\n Answer 2:\n{long_answers[1]}'
prompt = prompt_template.format(question=question, long_answers=long_answers)

#tokenize prompt
input_ids = tokenizer.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt')

out = model.generate(input_ids.to(device), max_new_tokens = 512)
res = tokenizer.decode(out[0]).split('<|end_header_id|>')[-1] 
candidate = [re.sub('\n|<\|eot_id\|>', '', res)]


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [31]:
from evaluate import evaluate

'''
candidate = list of generated long answers
references = list of imput as dictionary
'''

references = [row.to_dict() for i, row in df.iterrows() if i < len(candidate)]
print(references)

evaluate(candidate,references)

[{'id': '5cf39741-d5bd-4226-a94d-ff0c13ca5eaa', 'sample_id': -7013890438520559398, 'question': 'Who has the highest goals in world football?', 'follow_up_questions': '["Who has the highest goals in men\'s world international football?", "Who has the highest goals all-time in men\'s football?", "Who has the highest goals in women\'s world international football?"]', 'long_answers': '["Ali Dael has the highest goals in men\'s world international football with 109 goals. Josef Bican has the highest goals all-time in men\'s football and Christine Sinclair has the highest goals in women\'s world international football.", "The players with the highest all-time goals and highest men\'s and women\'s international football goals differ. The player with the highest all-time men\'s football goals is Josef Bican, who in 2020 was recognized by FIFA, the international governing body of football, as the record scorer with an estimated 805 goals. Christine Sinclair has the highest goals in women\'s in

{'rougeLsum': np.float64(54.08805031446542),
 'length': 52.0,
 'str_em': np.float64(100.0),
 'ovscore': np.float64(73.54457853197978)}